## Gemini Master Instructions for Modular Colab **Workflows**

You are helping design and maintain **modular Google Colab workflows** orchestrated by a **central master pipeline**. Every notebook step must remain self-contained, reusable, and easy to debug independently, while still fitting into a larger end-to-end workflow. Make sure to read and follow the guidance below.

---

### 1. Core Architecture Principles

* Build workflows as **modular processing blocks**.
* Each major step must be **self-contained** with clear inputs/outputs.
* Use a **single master pipeline block** to orchestrate execution.
* Avoid tight coupling between modules.
* Prefer explicit data handoffs (dataframes, return values, or defined globals).
* Design modules so they can be **tested independently**.

---

### 2. Variable Management Rules

* **Consolidate all variables within the master pipeline block.**
* If variables are needed elsewhere, **import them as globals**.
* Do not scatter configuration values across cells.
* Avoid duplicate constants inside modules.
* Add new variables to the master pipeline first, then wire downstream.

---

### 3. Notes and Commentary

* **Preserve all notes exactly where they are placed.**
* Do not remove or rewrite notes unless explicitly instructed.
* Treat markdown and comments as long-term documentation.
* Flag outdated notes instead of deleting them.

---

### 4. Notebook Structure

Preferred order:

1. Preflight / setup
2. Authentication / mounts
3. Shared imports
4. Module sections
5. Validation / diagnostics
6. Export / outputs
7. Master pipeline
8. Utility / recovery helpers

Use clear section headers (e.g., `### Preflight Check`, `# Master Pipeline`).

---

### 5. Module Design Requirements

* Each module should have **one responsibility**.
* Wrap logic in clearly named functions.
* Define inputs and outputs explicitly.
* Avoid hidden dependencies.
* Include lightweight validation.
* Document side effects (exports, file moves, etc.).

---

## 6. Master Pipeline Responsibilities

The master pipeline must:

* Define all variables and configuration
* Control execution order
* Pass configuration to modules
* Handle global state intentionally
* Manage logging and status
* Coordinate exports and failure handling

---

### 7. Globals Usage Rules

* Use globals only for intentionally shared configuration.
* Assign globals in the master pipeline.
* Avoid implicit globals in modules.
* Make dependencies on globals explicit.

---

### 8. Imports and Dependencies

* Keep imports organized.
* Avoid unnecessary duplication.
* Include imports in modules only if needed for isolation.
* Do not introduce unnecessary libraries.

---

### 9. Validation and Debugging

* Add validation checkpoints after transformations.
* Include diagnostics (row counts, schema checks, etc.).
* Print clear status messages.
* Fail gracefully where possible.

---

### 10. Output and Export Standards

* Keep export logic in a dedicated section.
* Use clear, traceable naming conventions.
* Avoid hidden output paths.
* Document outputs clearly.

---

### 11. Recovery and Utility Cells

* Keep utilities separate from core workflow.
* Clearly label recovery logic.
* Preserve existing utility cells.

---

### 12. Change Management

* Preserve structure unless improvement is necessary.
* Do not collapse modular design.
* Prefer targeted edits.
* Explain structural changes when needed.

---

### 13. Coding Style

* Write readable, maintainable code.
* Use clear function names.
* Prefer explicit logic over shortcuts.
* Preserve dataframe clarity.

---

### 14. Interaction Rules

When building workflows:

* Assume modular architecture is required.
* Place variables in the master pipeline.
* Preserve notes and structure.
* Return code ready for direct cell insertion.
* Highlight impacted sections when making changes.

---

### Short Instruction Block (Reusable)

```text
Build this Colab workflow using a modular notebook architecture.

Rules:
1. Consolidate all variables within the master pipeline block.
2. Import shared variables as globals when needed.
3. Preserve all notes and markdown exactly as placed.
4. Keep each module self-contained and reusable.
5. Use a master pipeline for orchestration.
6. Do not scatter configuration values.
7. Maintain clear section headers.
8. Separate validation, export, and recovery logic.
9. Prefer targeted updates over rewrites.
10. Write maintainable, debuggable code.
```


### DATA EXTRACTION LOGIC

Do not use hard-coded row numbers or fixed positional logic when parsing these reports. Instead, use anchor-based detection by defining constants for known header or label text, such as const headers = ['Occ (%)', 'Index (MPI)', ...], and locate rows dynamically based on those anchors. This ensures the parser remains stable even if rows shift between properties, report versions, or months.

The parsing logic should always identify the relevant section by searching for the expected text labels in the sheet, rather than assuming a metric will always appear on the same row. Build the mapping from those discovered anchor points, then derive the related values relative to the matched labels. This makes the pipeline more resilient and reduces breakage when report formatting changes.

Use anchor-based parsing only. Never rely on fixed row indexes for report extraction. Define reusable constants for expected labels and headers, scan the sheet to find those anchors, and build mappings from the discovered positions. This ensures the parser continues to work even when report layouts shift.

# Success Critera Test

To deterime if the merge was successful without affecting the underlaying data integrity this test made and must pass before considering success.

Display a Dataframe with the following totals, filtered by Date + Reservations Status for the exported file.

---

## Filters
Filters
Date = `01-01-2026` - `01-31-2026`
Reservation Status = `CHECKEDOUT`

---

## Sum Values
Sum Col `Sold`
Sum Col `Room Revenue`

---

## Criteria
Total for `Sum` must equal **2235**
Total for `Room` Revenue must equal **175872**

# Connect to Data

In [ ]:
# @title Connect to Google Drive {"vertical-output":true,"single-column":true,"display-mode":"code"}

from google.colab import drive
import os

def setup_environment(source_path, next_path):
    """
    Module: Setup Environment
    Mounts Google Drive, defines global directory variables, and ensures all
    required subfolders exist.
    """
    print("--- Initializing Environment ---")

    # 1. Mount Drive (with a safety catch)
    try:
        drive.mount('/content/drive', force_remount=True)
        print("v Drive mounted successfully.")
    except Exception as e:
        print(f"x Manual action required: Please click the Drive icon to mount. Error: {e}")

    # --- SETUP & AUTH ---
    # Use force_remount=True to attempt a fresh connection if it previously failed
    try:
        drive.mount('/content/drive', force_remount=True)
        print("Drive mounted successfully.")
    except Exception as e:
        print(f"Manual action required: Please click the Drive icon in the left file pane to mount your drive. Error: {e}")

    # --- CONFIGURATION (Master Variables) ---
    global SOURCE_DIR, NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR

    SOURCE_DIR = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step02" # @param {"type":"string","placeholder":"/content/drive/Shareddrives/Client Hubs/Dovetail&Co/data_pipeline/process_step02"}
    NEXT_DIR = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step03" # @param {"type":"string","placeholder":"/content/drive/Shareddrives/Client Hubs/Dovetail&Co/data_pipeline/process_step03"}

    NEW_DIR = os.path.join(SOURCE_DIR, "data_upload")
    PROCESSED_DIR = os.path.join(SOURCE_DIR, "data_processed")
    EXPORT_DIR = os.path.join(SOURCE_DIR, "data_export")
    FAILED_DIR = os.path.join(SOURCE_DIR, "data_failed")
    NEXT_DIR = os.path.join(NEXT_DIR, "data_upload")

    # Create the necessary folders if they don't exist
    if not os.path.exists(NEW_DIR):
        os.makedirs(NEW_DIR)
        print(f"Created directory: {NEW_DIR}")
    if not os.path.exists(PROCESSED_DIR):
        os.makedirs(PROCESSED_DIR)
        print(f"Created directory: {PROCESSED_DIR}")
    if not os.path.exists(EXPORT_DIR):
        os.makedirs(EXPORT_DIR)
        print(f"Created directory: {EXPORT_DIR}")
    if not os.path.exists(FAILED_DIR):
        os.makedirs(FAILED_DIR)
        print(f"Created directory: {FAILED_DIR}")

    # 4. Create directories dynamically if they don't exist
    directories = [NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR]

    for directory in directories:
        if not os.path.exists(directory):
            os.makedirs(directory)
            print(f"  -> Created directory: {directory}")

    print(f"v Setup complete. Checking for new files in: {NEW_DIR}\n")

# Start


In [ ]:
import glob
import os, shutil, re
from datetime import datetime
import pandas as pd
import numpy as np
from google.colab import drive

# --- SHARED HELPERS ---

def find_header_row(file_path, threshold=10):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if len(line.split(',')) > threshold: return i
    return 0

def load_latest_export(pattern_suffix, folder_path):
    search_pattern = os.path.join(folder_path, f"*{pattern_suffix}*.csv")
    files = [f for f in glob.glob(search_pattern) if os.path.isfile(f)]
    if not files:
        return None, None
    latest_file = max(files, key=os.path.getmtime)
    skip_count = find_header_row(latest_file)
    df = pd.read_csv(latest_file, skiprows=skip_count, engine='python')
    return df, latest_file

def load_data_assets():
    """Module to load required datasets."""
    global NEW_DIR, pms_df, crs_df, pms_file, crs_file

    print(f"🔍 Searching for files in: {NEW_DIR}")
    pms_df, pms_file = load_latest_export("_pms_data", NEW_DIR)
    crs_df, crs_file = load_latest_export("_crs_reservations", NEW_DIR)

    if pms_file and crs_file:
        print(f"✅ Found PMS: {os.path.basename(pms_file)}")
        print(f"✅ Found CRS: {os.path.basename(crs_file)}")
        return True
    else:
        if not pms_file: print("❌ Missing PMS file")
        if not crs_file: print("❌ Missing CRS file")
        return False

# --- EXECUTION ---
if load_data_assets():
    print(f"\nPMS Rows: {len(pms_df)} | CRS Rows: {len(crs_df)}")
    print("\n--- PMS Data Preview ---")
    display(pms_df.head())
    print("\n--- CRS Data Preview ---")
    display(crs_df.head())
else:
    print("\n⚠️ Loading failed. Check your data_upload folder.")

In [ ]:
def merge_pms_crs_data(pms_df, crs_df):
    """Core logic to clean IDs and merge CRS attributes into PMS data."""
    def clean_id(val):
        s = str(val).strip().lower()
        if s in ['nan', 'none', '', 'null']: return None
        if s.endswith('.0'): return s[:-2]
        return s

    pms_work = pms_df.copy()
    crs_work = crs_df.copy()

    pms_id_col = 'Confirmation Number' if 'Confirmation Number' in pms_work.columns else 'confirmation_number'
    crs_id_col = 'PMS_Confirm_Code' if 'PMS_Confirm_Code' in crs_work.columns else 'pms_conf'

    pms_work['match_id'] = pms_work[pms_id_col].apply(clean_id)
    crs_work['match_id'] = crs_work[crs_id_col].apply(clean_id)

    crs_cols = [
        'match_id', 'Confirm_No', 'Channel_Connect_Confirm_NO',
        'Channel_Cd', 'Sec_Channel_Desc', 'Sub_Src_CD', 'Sub_Source', 'Rate_Type_Code'
    ]

    crs_subset = crs_work.dropna(subset=['match_id'])[crs_cols].drop_duplicates(subset=['match_id'])
    crs_subset = crs_subset.rename(columns={
        'Confirm_No': 'crs_confirm_no',
        'Channel_Connect_Confirm_NO': 'channel_confirm_no',
        'Channel_Cd': 'crs_channel_code',
        'Sec_Channel_Desc': 'crs_channel',
        'Sub_Src_CD': 'crs_subsource_code',
        'Sub_Source': 'crs_subsource',
        'Rate_Type_Code': 'crs_rate_type'
    })

    merged = pd.merge(pms_work, crs_subset, on='match_id', how='left').drop(columns=['match_id'])

    fill_cols = ['crs_channel_code', 'crs_channel', 'crs_subsource_code', 'crs_subsource', 'crs_rate_type']
    for col in fill_cols:
        if col in merged.columns: merged[col] = merged[col].fillna('Unknown')

    return merged

def run_post_merge_verification():
    """Module to identify and verify merged columns dynamically."""
    global enriched_df
    if 'enriched_df' not in globals() or enriched_df is None:
        print("☑ No enriched data found for verification. Skipping diagnostic.")
        return

    print("၈ --- Post-Merge Column Verification ---")
    cols = enriched_df.columns.tolist()
    possible_market = [c for c in cols if 'Market' in c]
    possible_source = [c for c in cols if 'Source' in c]
    possible_rate = [c for c in cols if 'Rate' in c]

    verification_targets = []
    if possible_market: verification_targets.append(possible_market[0])
    if possible_source: verification_targets.append(possible_source[0])
    if possible_rate: verification_targets.append(possible_rate[0])
    if 'crs_channel' in cols: verification_targets.append('crs_channel')

    if verification_targets:
        display(enriched_df[verification_targets].head())
        for col in verification_targets:
            unique_vals = enriched_df[col].unique()[:5]
            print(f"- Sample values for {col}: {unique_vals}")

In [ ]:
def process_data_enrichment():
    """Module to orchestrate the enrichment of PMS data with CRS attributes."""
    global pms_df, crs_df, enriched_df

    print("⚙️ Starting Enrichment Module...")

    # Guard clause: Check if data is loaded
    if 'pms_df' not in globals() or pms_df is None or 'crs_df' not in globals() or crs_df is None:
        print("❌ Enrichment Failed: pms_df or crs_df is missing. Please run the loading module successfully first.")
        return False

    try:
        # Call the core logic defined in the processing block
        enriched_df = merge_pms_crs_data(pms_df, crs_df)

        print("✅ Enrichment logic executed.")
        display(enriched_df.head())
        return True
    except Exception as e:
        print(f"❌ Enrichment Module Failed: {e}")
        return False

# --- MODULE EXECUTION ---
process_data_enrichment()

In [ ]:
def validate_merge_quality(df):
    """Module for Validation & Diagnostics."""
    print("📊 --- Merge Diagnostics ---")
    total_rows = len(df)
    matched_rows = (df['crs_channel_code'] != 'Unknown').sum()
    match_rate = (matched_rows / total_rows) * 100

    print(f"Total Rows: {total_rows}")
    print(f"Successful CRS Matches: {matched_rows} ({match_rate:.2f}%)")

    print("\nMissing Data Heatmap (Top columns):")
    null_counts = df[['crs_channel', 'crs_subsource', 'crs_rate_type']].apply(lambda x: (x == 'Unknown').sum())
    print(null_counts)

# --- MODULE EXECUTION ---
if 'enriched_df' in globals():
    validate_merge_quality(enriched_df)
else:
    print("⚠️ No enriched data found to validate.")

In [ ]:
import pandas as pd
import os
import glob
from google.colab import drive
import pandas_gbq

# --- EXECUTE SETUP ---
# setup_environment definition is located in the Preflight section (Y66b4AMqXFwD)
setup_environment(
    source_path = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step02",
    next_path = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step03"
)

# --- MASTER PIPELINE ORCHESTRATION ---
def run_master_pipeline():
    global pms_file, crs_file, pms_df, crs_df, enriched_df

    print("ၣ Starting Master Pipeline...")

    # 1. Loading Module
    if not load_data_assets():
        print("ႀ Pipeline Stopped: Files not found or loading failed.")
        return False

    # 2. Enrichment Module
    if not process_data_enrichment():
        print("ႀ Pipeline Stopped: Enrichment failed.")
        return False

    # 3. Verification Module
    run_post_merge_verification()

    # 4. Export & File Management
    try:
        output_filename = os.path.basename(pms_file).replace('.csv', '_merged.csv')
        export_path = os.path.join(EXPORT_DIR, output_filename)
        next_step_path = os.path.join(NEXT_DIR, output_filename)

        enriched_df.to_csv(export_path, index=False)
        enriched_df.to_csv(next_step_path, index=False)
        print(f"၇ Exported merged file to: {output_filename}")

        # Archive Original Files (ONLY on Success Path)
        for f in [pms_file, crs_file]:
            if f and os.path.exists(f):
                dest = os.path.join(PROCESSED_DIR, os.path.basename(f))
                shutil.move(f, dest)
                print(f"ၦ Archived to Processed: {os.path.basename(f)}")
        return True

    except Exception as e:
        print(f"❌ Finalization Failure: {e}")
        for f in [pms_file, crs_file]:
            if f and os.path.exists(f):
                dest = os.path.join(FAILED_DIR, os.path.basename(f))
                shutil.move(f, dest)
                print(f"⚠ Moved to Failed: {os.path.basename(f)}")
        return False

# Execute Pipeline and Capture Success State
pipeline_success = run_master_pipeline()

# Triggers the next notebook in the pipeline ONLY if the entire pipeline returned True
if pipeline_success:
    print("🚀 Success: Triggering Step 03...")
    get_ipython().run_line_magic('run', "'/content/drive/MyDrive/Colab Notebooks/StayInTouch/Step03_StayInTouch_NormalizeHeaders.ipynb'")
else:
    print("🛑 Failure: Downstream pipeline was NOT triggered.")

In [ ]:
def run_post_merge_verification():
    """Module to identify and verify merged columns dynamically."""
    global enriched_df

    if 'enriched_df' not in globals() or enriched_df is None:
        print("⚠️ No enriched data found for verification. Skipping diagnostic.")
        return

    print("📊 --- Post-Merge Column Verification ---")
    cols = enriched_df.columns.tolist()

    # Dynamic detection based on substrings
    possible_market = [c for c in cols if 'Market' in c]
    possible_source = [c for c in cols if 'Source' in c]
    possible_rate = [c for c in cols if 'Rate' in c]

    verification_targets = []
    if possible_market: verification_targets.append(possible_market[0])
    if possible_source: verification_targets.append(possible_source[0])
    if possible_rate: verification_targets.append(possible_rate[0])
    if 'crs_channel' in cols: verification_targets.append('crs_channel')

    print(f"Detected verification columns: {verification_targets}")

    if verification_targets:
        display(enriched_df[verification_targets].head())
        # Print unique values for the first 5 target columns to verify enrichment diversity
        for col in verification_targets:
            unique_vals = enriched_df[col].unique()[:5]
            print(f"- Sample values for {col}: {unique_vals}")
    else:
        print("❌ Could not dynamically identify Market/Source/Rate columns.")

# --- MODULE EXECUTION ---
run_post_merge_verification()

# Validation / Diagnostics
This section verifies the integrity of the merged data against the success criteria defined for January 2026.

In [ ]:
import pandas as pd
import os
import glob

# --- Validation Module ---
def run_success_criteria_test(df):
    print("--- Running Success Criteria Test ---")

    # Filtering strictly on the 'Date' column as requested
    date_col = 'Date'

    if date_col not in df.columns:
         print(f"❌ Error: Required column '{date_col}' not found in the dataset.")
         return

    print(f"Filtering based on column: {date_col}")
    df[date_col] = pd.to_datetime(df[date_col])

    # 1. Define Filters (Criteria uhfOwqak033W)
    start_date = '2026-01-01'
    end_date = '2026-01-31'
    status_filter = 'CHECKEDOUT'

    # 2. Apply Filters
    mask = (
        (df[date_col] >= start_date) &
        (df[date_col] <= end_date) &
        (df['Reservation Status'] == status_filter)
    )
    test_df = df.loc[mask].copy()

    # 3. Calculate Sums
    total_sold = pd.to_numeric(test_df['Sold'], errors='coerce').sum()
    total_revenue = pd.to_numeric(test_df['Room Revenue'], errors='coerce').sum()

    # 4. Success Criteria Targets
    target_sold = 2235
    target_revenue = 175872

    # 5. Display Results with Tolerance for Decimals
    sold_pass = int(round(total_sold)) == target_sold
    rev_pass = abs(total_revenue - target_revenue) < 1.0

    results_data = {
        "Metric": ["Sum Col 'Sold'", "Sum Col 'Room Revenue'"],
        "Actual": [round(total_sold, 2), round(total_revenue, 2)],
        "Target": [target_sold, target_revenue],
        "Status": [
            "✅ PASS" if sold_pass else "❌ FAIL",
            "✅ PASS" if rev_pass else "❌ FAIL"
        ]
    }

    results_df = pd.DataFrame(results_data)
    display(results_df)

    if sold_pass and rev_pass:
        print("\n✨ SUCCESS: All criteria met. Data integrity confirmed.")
    else:
        print("\n⚠️ WARNING: Criteria mismatch. Check if the dates or filters need adjustment.")

# Execution Logic
if 'combined_df' in locals():
    run_success_criteria_test(combined_df)
elif 'EXPORT_DIR' in globals():
    files = glob.glob(os.path.join(EXPORT_DIR, "*_pms_data_merged.csv"))
    if files:
        latest_export = max(files, key=os.path.getmtime)
        print(f"Loading latest export for validation: {os.path.basename(latest_export)}")
        loaded_df = pd.read_csv(latest_export, low_memory=False)
        run_success_criteria_test(loaded_df)
    else:
        print("❌ Error: No exported files found in EXPORT_DIR.")
else:
    print("❌ Error: combined_df not found and EXPORT_DIR not defined.")